# On the `dual` function

The  JuMP `dual` function corresponds to a slightly different notion than what it is usually understood in linear programming, as illustrated below.

In [1]:
using JuMP
using LinearAlgebra
using HiGHS

Consider the following linear program.

In [2]:
m = Model()

@variable(m, x[1:3])

@constraint(m, c1, sum(x[i] for i = 1:2) >= 3)
@constraint(m, c2, -2x[1] + 2x[2] - 4x[3] <= 5)

@constraint(m, n2, x[2] >= 0)
@constraint(m, n3, x[3] <= 0)

@objective(m, Min, 4x[1]+2x[2]+x[3])

println(m)

Min 4 x[1] + 2 x[2] + x[3]


Subject to
 

c1 : x[1] + x[2] ≥ 3
 n2 : x[2] ≥ 0
 c2 : -2 x[1] + 2 x[2] - 4 x[3] ≤ 5
 n3 : x[3] ≤ 0



In [3]:
set_optimizer(m, HiGHS.Optimizer)

optimize!(m)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
LP has 4 rows; 3 cols; 7 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 4e+00]
  Cost    [1e+00, 4e+00]
  Bound   [0e+00, 0e+00]
  RHS     [3e+00, 5e+00]
Presolving model
2 rows, 3 cols, 5 nonzeros 0s
1 rows, 2 cols, 2 nonzeros 0s
0 rows, 0 cols, 0 nonzeros 0s
Presolve reductions: rows 0(-4); columns 0(-3); nonzeros 0(-7) - Reduced to empty
Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Objective value     :  6.5000000000e+00
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


In [4]:
value.(x)

3-element Vector{Float64}:
  0.25
  2.75
 -0.0

We can use the function `dual` to directly obtain the values of the dual variables, but as explained on the page https://jump.dev/JuMP.jl/stable/manual/constraints/ the objective function is not taken into account as the function uses the concept of conic duality: https://jump.dev/MathOptInterface.jl/v0.9.1/apimanual/#Duals-1

In the case of a minimization program, we obtain the desired result.

In [5]:
[dual(c1) ; dual(c2)]

2-element Vector{Float64}:
  3.0
 -0.5

JuMP also proposes the `shadow_price` function that evaluates the change in the objective if we relax the constraint by one unit. The term *relaxation* is not precisely defined, and we can see that on a greater-than constraint the sign is the opposite of what we expect.

In [6]:
[shadow_price(c1) ; shadow_price(c2)]

2-element Vector{Float64}:
 -3.0
 -0.5

Let's explicitly form the dual program in order to validate our observations.

In [7]:
m = Model()

@variable(m, y[1:2])

@constraint(m, c1, y[1] - 2y[2] == 4)
@constraint(m, c2, y[1] + 2y[2] <= 2)
@constraint(m, c3, -4y[2] >= 1)

@constraint(m, n1, y[1] >= 0)
@constraint(m, n2, y[2] <= 0)

@objective(m, Max, 3y[1]+5y[2])

println(m)

Max 3 y[1] + 5 y[2]
Subject to
 

c1 : y[1] - 2 y[2] = 4
 c3 : -4 y[2] ≥ 1
 n1 : y[1] ≥ 0
 c2 : y[1] + 2 y[2] ≤ 2
 n2 : y[2] ≤ 0



In [8]:
set_optimizer(m, HiGHS.Optimizer)

optimize!(m)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
LP has 5 rows; 2 cols; 7 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 4e+00]
  Cost    [3e+00, 5e+00]
  Bound   [0e+00, 0e+00]
  RHS     [1e+00, 4e+00]
Presolving model
0 rows, 0 cols, 0 nonzeros 0s
0 rows, 0 cols, 0 nonzeros 0s
Presolve reductions: rows 0(-5); columns 0(-2); nonzeros 0(-7) - Reduced to empty
Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Objective value     :  6.5000000000e+00
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


In [9]:
value.(y)

2-element Vector{Float64}:
  3.0
 -0.5

The problem is now a maximization one, and when we call the `dual` function, we get the opposite of the primal optimal solution.

In [10]:
[ dual(c1) ; dual(c2) ; dual(c3) ]

3-element Vector{Float64}:
 -0.25
 -2.75
  0.0

Here, the shadow prices correspond to what we expect, but the negative sign on the last variable reflects that the corresponding inequality constraint is of the type greater than.

In [11]:
[ shadow_price(c1) ; shadow_price(c2) ; shadow_price(c3) ]

3-element Vector{Float64}:
 0.25
 2.75
 0.0

We can also observe this behavior from the example given at https://jump.dev/JuMP.jl/stable/manual/constraints/

In [12]:
model = Model(HiGHS.Optimizer)
@variable(model, x)
@constraint(model, con, x <= 1)
@objective(model, Min, -2x)
optimize!(model)
dual(con)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
LP has 1 row; 1 col; 1 nonzero
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [2e+00, 2e+00]
  Bound   [0e+00, 0e+00]
  RHS     [1e+00, 1e+00]
Presolving model
0 rows, 0 cols, 0 nonzeros 0s
0 rows, 0 cols, 0 nonzeros 0s
Presolve reductions: rows 0(-1); columns 0(-1); nonzeros 0(-1) - Reduced to empty
Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Objective value     : -2.0000000000e+00
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


-2.0

In [13]:
shadow_price(con)

-2.0

In [14]:
 @objective(model, Max, 2x)

2 x

In [15]:
optimize!(model)

LP has 1 row; 1 col; 1 nonzero
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [2e+00, 2e+00]
  Bound   [0e+00, 0e+00]
  RHS     [1e+00, 1e+00]
Solving LP with useful basis so presolve not used

Model status        : Optimal
Objective value     :  2.0000000000e+00
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


In [16]:
dual(con)

-2.0

In [17]:
shadow_price(con)

2.0

In [18]:
model = Model(HiGHS.Optimizer)
@variable(model, x)
@constraint(model, con, -x >= 1)
@objective(model, Min, -2x)
optimize!(model)

Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: libblastrampoline 
LP has 1 row; 1 col; 1 nonzero
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [2e+00, 2e+00]
  Bound   [0e+00, 0e+00]
  RHS     [1e+00, 1e+00]
Presolving model
0 rows, 0 cols, 0 nonzeros 0s
0 rows, 0 cols, 0 nonzeros 0s
Presolve reductions: rows 0(-1); columns 0(-1); nonzeros 0(-1) - Reduced to empty
Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Objective value     :  2.0000000000e+00
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


In [19]:
dual(con)

2.0

In [20]:
shadow_price(con)

-2.0

In [21]:
@objective(model, Max, 2x)
optimize!(model)

LP has 1 row; 1 col; 1 nonzero
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [2e+00, 2e+00]
  Bound   [0e+00, 0e+00]
  RHS     [1e+00, 1e+00]
Solving LP with useful basis so presolve not used

Model status        : Optimal
Objective value     : -2.0000000000e+00
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


In [22]:
dual(con)

2.0

In [23]:
shadow_price(con)

2.0